# 01 · 원본 전처리

**00 실행 후** 이 노트북을 실행하세요. 원본 파일을 읽어 관측 거래와 라벨을 분리하고,
거래 17차원 캐시(학습에 14개 사용), 결제형식·Port, 계좌 19차원 일별 스냅샷을 생성합니다.
라벨·패턴은 정답과 평가에만 사용하며 피처·그래프 구성에서 제외합니다.

1. 원본 CSV 정리: 원본 행 ID 유지, `(은행, 계좌)` 노드 매핑, 관측 필드 기준 중복 제거, 패턴 매칭.
2. 거래 피처: 같은 분을 제외한 과거 거래 수·간격, 당일 시작 전의 역방향 이력, 금액 정규화·KYC.
3. 계좌 피처: 일별 누적 거래·이웃·상호거래·통화·활동일·소유주 통계. 다음 날 입력은 참조하지 않습니다.

완료한 전처리 단계는 같은 입력·코드·설정에서 재실행하면 건너뜁니다. 원본 또는 코드가 바뀌면 새 WORK_DIR를 사용하세요.
원본 변경 감지는 크기·수정시각·앞뒤 1 MiB 해시를 사용합니다. 전체 원본 내용의 SHA256 검사는 아닙니다.

In [ ]:
from pathlib import Path
import json
import sys
import subprocess

PROJECT_DIR = Path.cwd().resolve()
link = PROJECT_DIR / ".salt_project.json"
if not link.exists():
    raise FileNotFoundError("이 노트북들이 있는 폴더에서 00_환경설정과_실행준비.ipynb를 먼저 실행하세요.")
project = json.loads(link.read_text())
WORK_DIR = Path(project["work_dir"])
DEVICE = project["device"]
for module_name in ("salt_daily", "runtime", "run", "notebook_control"):
    loaded = sys.modules.get(module_name)
    location = getattr(loaded, "__file__", None)
    if location and not Path(location).resolve().is_relative_to(WORK_DIR):
        raise RuntimeError("다른 WORK_DIR가 import되어 있습니다. 커널을 재시작하세요.")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

def command(stage, device=DEVICE):
    subprocess.run([sys.executable, "-u", str(WORK_DIR/"run.py"), stage, "--device", device], cwd=WORK_DIR, check=True)

print("작업 폴더:", WORK_DIR)


### 이 단계의 실행 코드

아래 소스는 00 노트북이 읽어 작업 폴더에 실행 파일로 만듭니다. 코드를 수정하면 노트북을 저장하고 **새 WORK_DIR**로 00을 다시 실행하세요.

In [ ]:
SOURCES = {}


In [ ]:
# salt_daily/raw_common.py
SOURCES['salt_daily/raw_common.py'] = '''"""IBM AML CSV -> label-independent features, sparse labels and cumulative graphs.

DuckDB performs joins, windows and external sorts; NumPy outputs are written in
bounded Arrow batches. This entry point never imports Torch or starts training.
"""

import csv

from pathlib import Path

import duckdb
import numpy as np


from .raw_config import ENTITY_TYPES, FORMAT_NAMES


RAW_COLUMNS = (
    "timestamp_raw",
    "from_bank",
    "from_account",
    "to_bank",
    "to_account",
    "amount_received_raw",
    "receiving_currency",
    "amount_paid_raw",
    "payment_currency",
    "payment_format",
    "label_raw",
)
HEADERS = (
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
)
ACCOUNT_COLUMNS = ("bank_name", "bank_id", "account_number", "entity_id", "entity_name")
ACCOUNT_HEADERS = ("Bank Name", "Bank ID", "Account Number", "Entity ID", "Entity Name")
OBSERVATION_KEY = (
    "ts",
    "src",
    "dst",
    "paid",
    "received",
    "payment_currency",
    "receiving_currency",
    "payment_format",
)


def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"


def csv_reader(path, columns, headers):
    with Path(path).open(newline="", encoding="utf-8-sig") as stream:
        if tuple(next(csv.reader(stream))) != tuple(headers):
            raise ValueError(f"Unexpected CSV header: {path}")
    schema = "{" + ",".join(f"{sql_string(name)}: 'VARCHAR'" for name in columns) + "}"
    return (
        f"read_csv({sql_string(path)}, header=true, columns={schema}, "
        "parallel=false, auto_detect=false, delim=',', quote='\\"', escape='\\"')"
    )


def export_array(
    connection, query, path, shape, dtype, chunk_rows=250_000, indexed=False
):
    """The indexed form expects edge/node ID as its first selected column."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    array = np.lib.format.open_memmap(path, mode="w+", dtype=dtype, shape=shape)
    cursor = 0
    for batch in connection.execute(query).to_arrow_reader(chunk_rows):
        columns = [column.to_numpy(zero_copy_only=False) for column in batch.columns]
        positions = (
            columns.pop(0).astype(np.int64)
            if indexed
            else slice(cursor, cursor + len(batch))
        )
        values = columns[0] if len(shape) == 1 else np.column_stack(columns)
        if not np.isfinite(values).all():
            raise ValueError(f"Nonfinite input to {path.name}")
        array[positions] = values
        cursor += len(batch)
    if not indexed and cursor != shape[0]:
        raise ValueError(f"Row count mismatch for {path.name}: {cursor} != {shape[0]}")
    array.flush()


def ingest(connection, transactions, accounts, cfg):
    """Original file row ID survives observed-field deduplication."""
    c = connection
    c.execute(
        f"CREATE TABLE raw AS SELECT row_number() OVER ()-1 edge_id, * FROM "
        f"{csv_reader(transactions, RAW_COLUMNS, HEADERS)}"
    )
    invalid = c.execute("""SELECT count(*) FROM raw WHERE
        try_strptime(timestamp_raw, '%Y/%m/%d %H:%M') IS NULL OR
        try_cast(from_bank AS BIGINT) IS NULL OR try_cast(to_bank AS BIGINT) IS NULL OR
        NOT regexp_full_match(from_account, '[0-9A-Fa-f]+') OR
        NOT regexp_full_match(to_account, '[0-9A-Fa-f]+') OR
        try_cast(amount_paid_raw AS DECIMAL(38,8)) IS NULL OR
        try_cast(amount_received_raw AS DECIMAL(38,8)) IS NULL OR
        try_cast(amount_paid_raw AS DECIMAL(38,8)) < 0 OR
        try_cast(amount_received_raw AS DECIMAL(38,8)) < 0 OR
        label_raw NOT IN ('0','1')""").fetchone()[0]
    empty = " OR ".join(f"{name} IS NULL OR trim({name})=''" for name in RAW_COLUMNS)
    if invalid or c.execute(f"SELECT count(*) FROM raw WHERE {empty}").fetchone()[0]:
        raise ValueError(
            "Invalid transaction rows; fix the source before preprocessing"
        )
    c.execute(
        """CREATE TABLE nodes AS SELECT row_number() OVER(ORDER BY bank, account)-1 node_id,
        bank, account FROM (
        SELECT cast(cast(from_bank AS BIGINT) AS VARCHAR) bank, from_account account FROM raw
        UNION SELECT cast(cast(to_bank AS BIGINT) AS VARCHAR), to_account FROM raw)"""
    )
    c.execute(
        f"CREATE TABLE accounts AS SELECT * FROM {csv_reader(accounts, ACCOUNT_COLUMNS, ACCOUNT_HEADERS)}"
    )
    if c.execute("""SELECT count(*) FROM (SELECT bank_id,account_number FROM accounts
        GROUP BY bank_id,account_number HAVING count(*)>1)""").fetchone()[0]:
        raise ValueError("Account lookup must contain a unique bank/account key")
    c.execute(
        """CREATE TABLE kyc AS SELECT n.*, a.entity_id,
        coalesce(regexp_extract(a.entity_name, '^([A-Za-z ]+?)\\\\s*#', 1),'') entity_type,
        CASE WHEN lower(a.bank_name) LIKE '%crypto%' OR lower(a.bank_name) LIKE '%crytpo%'
             THEN '' ELSE coalesce(regexp_extract(a.bank_name, '^(.*?) Bank #', 1),'') END country
        FROM nodes n LEFT JOIN accounts a ON
        n.bank=cast(try_cast(a.bank_id AS BIGINT) AS VARCHAR) AND n.account=a.account_number"""
    )
    c.execute("""CREATE TABLE typed AS SELECT r.edge_id,
        strptime(timestamp_raw,'%Y/%m/%d %H:%M') ts, s.node_id src, d.node_id dst,
        cast(amount_paid_raw AS DECIMAL(38,8)) paid,
        cast(amount_received_raw AS DECIMAL(38,8)) received,
        payment_currency, receiving_currency, payment_format, cast(label_raw AS TINYINT) y
        FROM raw r JOIN nodes s ON s.bank=cast(cast(r.from_bank AS BIGINT) AS VARCHAR)
        AND s.account=r.from_account JOIN nodes d ON
        d.bank=cast(cast(r.to_bank AS BIGINT) AS VARCHAR) AND d.account=r.to_account""")
    key = ",".join(OBSERVATION_KEY)
    if c.execute(
        f"SELECT count(*) FROM (SELECT {key} FROM typed GROUP BY {key} "
        "HAVING min(y)<>max(y))"
    ).fetchone()[0]:
        raise ValueError("Duplicate observations have conflicting labels")
    # Deduplication key excludes labels; genuine nonidentical repeat transfers remain.
    c.execute(f"""CREATE TABLE tx AS SELECT * FROM typed
        QUALIFY row_number() OVER(PARTITION BY {key} ORDER BY edge_id)=1""")
    c.execute(
        f"""CREATE VIEW observed AS SELECT edge_id,{key},cast(ts AS DATE) AS day FROM tx
        WHERE ts >= TIMESTAMP {sql_string(cfg.start)} AND ts < TIMESTAMP {sql_string(cfg.end)}"""
    )


def build_transaction_features(c, cfg):
    """Causal minute windows, prior-day reverse history and frozen amount scaling."""
    c.execute(f"""CREATE TABLE calibration AS
        SELECT payment_currency currency,cast(paid AS DOUBLE) amount FROM observed
        WHERE ts < TIMESTAMP {sql_string(cfg.calibration_end)} UNION ALL
        SELECT receiving_currency,cast(received AS DOUBLE) FROM observed
        WHERE ts < TIMESTAMP {sql_string(cfg.calibration_end)}""")
    c.execute(
        """CREATE TABLE scales AS SELECT currency,
        coalesce(median(amount) FILTER(WHERE amount>0),1.) scale FROM calibration GROUP BY currency"""
    )
    c.execute("""CREATE TABLE robust AS SELECT currency,scale,median(z) med,
        greatest(quantile_cont(z,.75)-quantile_cont(z,.25),.001) iqr FROM
        (SELECT currency,scale,ln(1+amount/scale) z FROM calibration JOIN scales USING(currency))
        GROUP BY currency,scale""")
    # Same-minute transactions do not see each other in temporal input columns.
    c.execute(
        """CREATE TABLE windows AS SELECT edge_id,
        count(*) OVER pair_all pair_count, max(ts) OVER pair_all pair_last,
        count(*) OVER pair_hour pair_hour_count,
        max(ts) OVER sender_all sender_last, count(*) OVER sender_hour sender_hour_count,
        max(ts) OVER receiver_all receiver_last, count(*) OVER receiver_hour receiver_hour_count
        FROM observed WINDOW
        pair_all AS (PARTITION BY src,dst ORDER BY ts RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL 1 MINUTE PRECEDING),
        pair_hour AS (PARTITION BY src,dst ORDER BY ts RANGE BETWEEN INTERVAL 60 MINUTE PRECEDING AND INTERVAL 1 MINUTE PRECEDING),
        sender_all AS (PARTITION BY src ORDER BY ts RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL 1 MINUTE PRECEDING),
        sender_hour AS (PARTITION BY src ORDER BY ts RANGE BETWEEN INTERVAL 60 MINUTE PRECEDING AND INTERVAL 1 MINUTE PRECEDING),
        receiver_all AS (PARTITION BY dst ORDER BY ts RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL 1 MINUTE PRECEDING),
        receiver_hour AS (PARTITION BY dst ORDER BY ts RANGE BETWEEN INTERVAL 60 MINUTE PRECEDING AND INTERVAL 1 MINUTE PRECEDING)"""
    )
    c.execute("""CREATE TABLE pair_days AS SELECT src,dst,day,max(ts) last_ts,
        sum(count(*)) OVER(PARTITION BY src,dst ORDER BY day) cumulative
        FROM observed GROUP BY src,dst,day""")
    c.execute("""CREATE TABLE history AS SELECT t.edge_id,p.last_ts pair_last_day,
        coalesce(r.cumulative,0) reverse_count FROM observed t
        ASOF LEFT JOIN pair_days p ON t.src=p.src AND t.dst=p.dst AND t.day>p.day
        ASOF LEFT JOIN pair_days r ON t.src=r.dst AND t.dst=r.src AND t.day>r.day""")
    fmt = (
        "CASE t.payment_format "
        + " ".join(
            f"WHEN {sql_string(name)} THEN {i}"
            for i, name in enumerate(FORMAT_NAMES[:-1])
        )
        + " ELSE 7 END"
    )

    def gap(name, end="t.ts"):
        return f"CASE WHEN {name} IS NULL THEN 0 ELSE ln(2+date_diff('minute',{name},{end})) END"

    c.execute(f"""CREATE TABLE features AS SELECT t.edge_id,t.src,t.dst,
        date_diff('minute',TIMESTAMP {sql_string(cfg.start)},t.ts) AS minute,{fmt} AS format,
        ln(1+w.pair_count) f0,{gap('w.pair_last')} f1,ln(1+w.pair_hour_count) f2,
        {gap('w.sender_last')} f3,ln(1+w.sender_hour_count) f4,
        {gap('w.receiver_last')} f5,ln(1+w.receiver_hour_count) f6,
        CASE WHEN r.currency IS NULL THEN 0 ELSE
          greatest(-8,least(8,(ln(1+cast(t.paid AS DOUBLE)/r.scale)-r.med)/r.iqr)) END f7,
        ln(1+h.reverse_count) f8,{gap('h.pair_last_day', 'cast(t.day AS TIMESTAMP)')} f9,
        (s.country<>'' AND d.country<>'' AND s.country=d.country)::INTEGER f10,
        (s.country<>'' AND d.country<>'')::INTEGER f11,
        (t.payment_currency=t.receiving_currency)::INTEGER f12,(t.src=t.dst)::INTEGER f13
        FROM observed t JOIN windows w USING(edge_id) JOIN history h USING(edge_id)
        LEFT JOIN robust r ON t.payment_currency=r.currency
        JOIN kyc s ON s.node_id=t.src JOIN kyc d ON d.node_id=t.dst""")
    c.execute("""CREATE TABLE first_pair AS SELECT src,dst,min(ts) first_ts,
        arg_min(edge_id,(ts,edge_id)) first_id FROM observed GROUP BY src,dst""")
    c.execute(
        """CREATE TABLE ports AS SELECT src,dst,
        row_number() OVER(PARTITION BY dst ORDER BY first_ts,first_id)-1 in_port,
        row_number() OVER(PARTITION BY src ORDER BY first_ts,first_id)-1 out_port FROM first_pair"""
    )


def build_node_snapshot(c, root, scope, end, nodes, chunk_rows):
    c.execute(
        f"CREATE OR REPLACE TEMP VIEW prefix AS SELECT t.*,f.f7 paid_z FROM observed t "
        f"JOIN features f USING(edge_id) WHERE ts<TIMESTAMP {sql_string(end)}"
    )
    c.execute(
        """CREATE OR REPLACE TEMP TABLE pairs AS SELECT DISTINCT src,dst FROM prefix"""
    )
    one_hot = ",".join(
        f"(k.entity_type={sql_string(name)})::INTEGER" for name in ENTITY_TYPES
    )
    query = f"""WITH
        out_stats AS (SELECT src node_id,count(*) nout,count(DISTINCT dst) peers,
            count(*) FILTER(WHERE src=dst) self_count,avg(paid_z) z,
            avg((abs(paid_z)>=3)::INTEGER) tail FROM prefix GROUP BY src),
        in_stats AS (SELECT dst node_id,count(*) nin,count(DISTINCT src) peers FROM prefix GROUP BY dst),
        reciprocal AS (SELECT a.src node_id,count(*) n FROM pairs a JOIN pairs b
            ON a.src=b.dst AND a.dst=b.src WHERE a.src<>a.dst GROUP BY a.src),
        currency AS (SELECT node_id,count(DISTINCT currency) n FROM (
            SELECT src node_id,payment_currency currency FROM prefix UNION ALL
            SELECT dst,receiving_currency FROM prefix) GROUP BY node_id),
        active AS (SELECT node_id,count(DISTINCT day) n FROM (
            SELECT src node_id,day FROM prefix UNION ALL SELECT dst,day FROM prefix) GROUP BY node_id),
        owner_nodes AS (SELECT k.entity_id,count(*) n FROM kyc k JOIN active a USING(node_id)
            WHERE k.entity_id IS NOT NULL GROUP BY k.entity_id),
        owner_pairs AS (SELECT DISTINCT s.entity_id source_owner,d.entity_id target_owner
            FROM pairs p JOIN kyc s ON p.src=s.node_id JOIN kyc d ON p.dst=d.node_id
            WHERE s.entity_id IS NOT NULL AND d.entity_id IS NOT NULL),
        owner_out AS (SELECT source_owner entity_id,count(*) n FROM owner_pairs GROUP BY source_owner),
        owner_in AS (SELECT target_owner entity_id,count(*) n FROM owner_pairs GROUP BY target_owner)
        SELECT k.node_id, ln(1+coalesce(o.nout,0)),ln(1+coalesce(i.nin,0)),
            ln(1+coalesce(o.peers,0)),ln(1+coalesce(i.peers,0)),
            ln(1+coalesce(o.self_count,0)),ln(1+coalesce(r.n,0)),
            coalesce(o.z,0),coalesce(o.tail,0),ln(1+c.n),ln(1+a.n),
            {one_hot},ln(1+coalesce(own.n,0)),ln(1+coalesce(oo.n,0)),ln(1+coalesce(oi.n,0))
        FROM kyc k JOIN active a USING(node_id) LEFT JOIN out_stats o USING(node_id)
        LEFT JOIN in_stats i USING(node_id) LEFT JOIN reciprocal r USING(node_id)
        JOIN currency c USING(node_id) LEFT JOIN owner_nodes own USING(entity_id)
        LEFT JOIN owner_out oo USING(entity_id) LEFT JOIN owner_in oi USING(entity_id)"""
    # open_memmap initializes unwritten/inactive nodes to zero.
    export_array(
        c,
        query,
        root / f"nodes_{scope}.npy",
        (nodes, 19),
        "float32",
        chunk_rows,
        indexed=True,
    )

'''


In [ ]:
# salt_daily/preprocessing.py
SOURCES['salt_daily/preprocessing.py'] = '''"""Raw CSV -> observed transactions, labels and daily 19/17 feature caches.

All feature queries use the observed view, which contains no labels or patterns.
Stages commit independently and never modify raw files or an existing run.
"""
import csv
import gc
import hashlib
import json
import fcntl
import uuid
from dataclasses import asdict
from datetime import date, timedelta
from pathlib import Path
from contextlib import contextmanager

import duckdb
import numpy as np
import pyarrow.parquet as pq

from .config import ROOT, DATA_ROOT, RUN
from .storage import read_json, write_json, sha, utc
from .raw_config import RawConfig, ENTITY_TYPES, PATTERN_NAMES
from .raw_common import (ingest, build_transaction_features, export_array,
                         sql_string as q, csv_reader, RAW_COLUMNS, HEADERS)


def config():
    return RawConfig(**read_json(ROOT/'raw_settings.json')['preprocess'])


def fingerprint(path):
    path = Path(path).expanduser().resolve()
    s = path.stat()
    with path.open('rb') as f:
        first = f.read(1024**2)
        f.seek(max(0, s.st_size-1024**2))
        last = f.read()
    return dict(path=str(path), bytes=s.st_size, mtime_ns=s.st_mtime_ns,
                head_tail_sha256=hashlib.sha256(first+last).hexdigest())


def identity():
    settings = read_json(ROOT/'raw_settings.json')
    return dict(config=asdict(config()), sources={k:fingerprint(v) for k,v in settings['files'].items()},
        code={p.name:sha(p) for p in sorted((ROOT/'salt_daily').glob('raw_*.py'))},
        preprocessing_sha256=sha(Path(__file__)), graph_sha256=sha(ROOT/'salt_daily/graph_prepare.py'), duckdb=duckdb.__version__)


@contextmanager
def stage(name, dependencies=()):
    folder=DATA_ROOT/'_build';folder.mkdir(parents=True,exist_ok=True)
    RUN.mkdir(parents=True,exist_ok=True)
    with (folder/'prepare.lock').open('a') as lock, (RUN/'run.lock').open('a') as run_lock:
        fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
        fcntl.flock(run_lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
        request=identity(); marker=folder/(name+'.json')
        for dep in dependencies:
            previous=read_json(folder/(dep+'.json'))
            if previous['identity']!=request or previous['status']!='complete':
                raise ValueError('Preprocessing inputs/code changed; use a new WORK_DIR')
        if marker.exists():
            old=read_json(marker)
            if old['identity']!=request or old['status']!='complete':
                raise ValueError('Existing preprocessing stage differs; use a new WORK_DIR')
            print(name, 'already complete', flush=True)
            yield None
            return
        c=duckdb.connect(str(folder/'raw.duckdb'))
        cfg=config()
        c.execute('SET memory_limit='+q(cfg.memory_limit))
        c.execute(f'SET threads={cfg.threads}')
        c.execute('SET temp_directory='+q(folder/'spill'))
        c.execute('SET preserve_insertion_order=true')
        try:
            print(name,'started',flush=True)
            c.execute('BEGIN TRANSACTION')
            yield c
            if identity()!=request:
                raise RuntimeError('Raw input changed during preprocessing')
            c.execute('COMMIT'); c.execute('CHECKPOINT')
            write_json(marker,dict(status='complete',identity=request,completed_utc=utc()))
            print(name,'complete',flush=True)
        except BaseException:
            try:c.execute('ROLLBACK')
            except duckdb.Error:pass
            raise
        finally:c.close()


def build_pattern_labels(c):
    source=Path(read_json(ROOT/'raw_settings.json')['files']['patterns'])
    target=DATA_ROOT/'_build/pattern_rows.csv'
    names=RAW_COLUMNS+('pattern_class',)
    headers=HEADERS+('Pattern Class',)
    current=None; rows=0
    with source.open(encoding='utf-8-sig') as f,target.open('w',newline='') as out:
        writer=csv.writer(out);writer.writerow(headers)
        for number,line in enumerate(f,1):
            line=line.strip()
            if not line:continue
            if line.startswith('BEGIN LAUNDERING ATTEMPT'):
                if current is not None:raise ValueError(f'Nested pattern block at {number}')
                name=line.split(' - ',1)[1].split(':',1)[0].strip()
                if name not in PATTERN_NAMES[1:9]:raise ValueError(f'Unknown pattern: {name}')
                current=PATTERN_NAMES.index(name)
            elif line.startswith('END LAUNDERING ATTEMPT'):
                if current is None:raise ValueError(f'Unmatched pattern END at {number}')
                current=None
            else:
                values=next(csv.reader([line]))
                if tuple(values)==HEADERS:continue
                if current is None or len(values)!=11 or values[-1]!='1':
                    raise ValueError(f'Invalid pattern transaction at {number}')
                writer.writerow(values+[current]);rows+=1
    if current is not None:raise ValueError('Unclosed pattern block')
    c.execute(f'CREATE TABLE patterns AS SELECT DISTINCT * FROM {csv_reader(target,names,headers)}')
    condition=' AND '.join(f'r.{name}=p.{name}' for name in RAW_COLUMNS)
    missing=c.execute(f'SELECT count(*) FROM patterns p WHERE NOT EXISTS (SELECT 1 FROM raw r WHERE {condition})').fetchone()[0]
    if missing:raise ValueError(f'{missing} pattern rows do not match the original CSV')
    c.execute(f'CREATE TABLE known AS SELECT DISTINCT r.edge_id,cast(p.pattern_class AS TINYINT) actual_class FROM raw r JOIN patterns p ON {condition}')
    if c.execute('SELECT count(*) FROM (SELECT edge_id FROM known GROUP BY edge_id HAVING count(DISTINCT actual_class)>1)').fetchone()[0]:
        raise ValueError('One transaction has conflicting pattern classes')
    c.execute(\'\'\'CREATE TABLE positives AS SELECT t.edge_id,coalesce(k.actual_class,9)::TINYINT actual_class
        FROM tx t JOIN observed o USING(edge_id) LEFT JOIN known k USING(edge_id) WHERE t.y=1\'\'\')
    return rows


def ingest_raw():
    with stage('01_ingest') as c:
        if c is None:return
        settings=read_json(ROOT/'raw_settings.json')['files']
        ingest(c,settings['transactions'],settings['accounts'],config())
        patterns=build_pattern_labels(c)
        nodes=c.execute('SELECT count(*) FROM nodes').fetchone()[0]
        rows,width=c.execute('SELECT count(*),max(edge_id)+1 FROM observed').fetchone()
        if not (0<width<2**31 and 0<nodes<2**31):raise ValueError('Native sampler requires int32 IDs')
        columns={r[0] for r in c.execute('DESCRIBE observed').fetchall()}
        assert not columns.intersection({'y','label_raw','pattern_class','actual_class'})
        folder=DATA_ROOT/'labels/assets';folder.mkdir(parents=True,exist_ok=True)
        n=c.execute('SELECT count(*) FROM positives').fetchone()[0]
        for field,name,dtype in [('edge_id','positive_ids','uint32'),('actual_class','positive_classes','int8')]:
            export_array(c,f'SELECT {field} FROM positives ORDER BY edge_id',folder/(name+'.npy'),(n,),dtype,config().chunk_rows)
        write_json(DATA_ROOT/'raw_metadata.json',dict(nodes=nodes,width=width,rows=rows,positive=n,
            pattern_rows=patterns,pattern_names=PATTERN_NAMES,origin='fresh_raw_csv',historical_cache_parity_verified=False,
            start=config().start,end_exclusive=config().end,label_free_observed_columns=sorted(columns)))


def build_edges():
    with stage('02_edges',('01_ingest',)) as c:
        if c is None:return
        cfg=config();meta=read_json(DATA_ROOT/'raw_metadata.json');width=meta['width']
        build_transaction_features(c,cfg)
        c.execute(\'\'\'CREATE TABLE amount_stats AS SELECT payment_currency,
            avg(ln(1+paid::DOUBLE)) m,stddev_pop(ln(1+paid::DOUBLE)) sd
            FROM observed WHERE day<DATE '2022-08-08' GROUP BY payment_currency\'\'\')
        c.execute(\'\'\'CREATE TABLE salt_features AS SELECT f.*,
            ln(1+coalesce(p.cumulative,0)) pair_history,
            (s.entity_id IS NOT NULL AND s.entity_id=d.entity_id)::INTEGER same_entity,
            CASE WHEN a.sd IS NULL THEN 0 ELSE greatest(-10,least(10,(ln(1+t.paid::DOUBLE)-a.m)/(a.sd+0.000001))) END amount_z
            FROM features f JOIN observed t USING(edge_id)
            ASOF LEFT JOIN pair_days p ON t.src=p.src AND t.dst=p.dst AND t.day>p.day
            JOIN kyc s ON s.node_id=t.src JOIN kyc d ON d.node_id=t.dst
            LEFT JOIN amount_stats a USING(payment_currency)\'\'\')
        for name,field,dtype in [('src','src','int32'),('dst','dst','int32'),('minute','minute','int32'),
                                ('fmt','format','int8'),('z','amount_z','float32')]:
            export_array(c,f'SELECT edge_id,{field} FROM salt_features',DATA_ROOT/'features'/(name+'.npy'),(width,),dtype,cfg.chunk_rows,True)
        for folder in ('features','structural'):
            export_array(c,'SELECT edge_id,1 FROM observed',DATA_ROOT/folder/'present.npy',(width,),'uint8',cfg.chunk_rows,True)
        columns=[f'f{i}' for i in range(8)]+['pair_history','f8','f9','same_entity','f10','f11','f12','f13','amount_z']
        export_array(c,'SELECT edge_id,'+','.join(columns)+' FROM salt_features',DATA_ROOT/'structural/x.npy',(width,17),'float32',cfg.chunk_rows,True)
        export_array(c,'SELECT edge_id,f7 FROM features',DATA_ROOT/'graph/hi/paid_z.npy',(width,),'float32',cfg.chunk_rows,True)
        for field,name in [('in_port','incoming_ports'),('out_port','outgoing_ports')]:
            export_array(c,f'SELECT edge_id,{field} FROM observed JOIN ports USING(src,dst)',DATA_ROOT/'ports'/(name+'.npy'),(width,),'int32',cfg.chunk_rows,True)


def partition(c,query,name,key):
    folder=DATA_ROOT/'_build'/name
    if (folder/'complete.json').exists():return folder
    if folder.exists():folder.rename(folder.with_name(name+'.incomplete.'+uuid.uuid4().hex[:8]))
    c.execute(f'COPY ({query}) TO {q(folder)} (FORMAT PARQUET,COMPRESSION ZSTD,PARTITION_BY({key}))')
    folder.mkdir(parents=True,exist_ok=True)
    write_json(folder/'complete.json',dict(query=query,key=key))
    return folder


def day_values(folder,key,day,columns):
    path=folder/f'{key}={day}'
    if not path.exists():return {name:np.empty(0,np.int64) for name in columns}
    table=pq.read_table(path,columns=columns)
    return {name:table[name].to_numpy(zero_copy_only=False) for name in columns}


def build_daily_nodes():
    with stage('03_daily_nodes',('01_ingest','02_edges')) as c:
        if c is None:return
        cfg=config();meta=read_json(DATA_ROOT/'raw_metadata.json');n=meta['nodes']
        k=c.execute(\'\'\'WITH owners AS (SELECT entity_id,(row_number() OVER(ORDER BY entity_id)-1)::INTEGER AS owner_code
              FROM (SELECT DISTINCT entity_id FROM kyc WHERE entity_id IS NOT NULL))
            SELECT k.node_id,coalesce(o.owner_code,-1) AS owner_code,k.entity_type FROM kyc k LEFT JOIN owners o USING(entity_id) ORDER BY node_id\'\'\').df()
        owner=k.owner_code.to_numpy(np.int64);types=k.entity_type.map({v:i for i,v in enumerate(ENTITY_TYPES)}).fillna(-1).to_numpy(np.int8)
        ne=int(owner.max())+1
        currencies=[r[0] for r in c.execute('SELECT payment_currency FROM observed UNION SELECT receiving_currency FROM observed ORDER BY 1').fetchall()]
        if len(currencies)>64:raise ValueError('This cache format supports at most 64 distinct currencies')
        def currency_code(column):
            return 'CASE '+column+' '+ ' '.join(f'WHEN {q(v)} THEN {i}' for i,v in enumerate(currencies))+' ELSE -1 END'
        raw=partition(c,f\'\'\'SELECT t.edge_id eid,t.day,t.src,t.dst,f.f7 paid_z,
            ({currency_code('t.payment_currency')})::TINYINT pc,({currency_code('t.receiving_currency')})::TINYINT rc
            FROM observed t JOIN features f USING(edge_id)\'\'\','raw_days','day')
        first=partition(c,'SELECT src,dst,min(day) first_day FROM observed GROUP BY src,dst','pair_first','first_day')
        pairs=f'read_parquet({q(str(first)+"/**/*.parquet")},hive_partitioning=true)'
        reciprocal=partition(c,f\'\'\'SELECT a.src node,greatest(a.first_day,b.first_day)::DATE first_day,count(*) n
            FROM {pairs} a JOIN {pairs} b ON a.src=b.dst AND a.dst=b.src WHERE a.src<>a.dst
            GROUP BY a.src,greatest(a.first_day,b.first_day)\'\'\','reciprocal_first','first_day')
        c.register('numeric_owners',k[['node_id','owner_code']])
        entity=partition(c,f\'\'\'SELECT a.owner_code source_owner,b.owner_code target_owner,min(p.first_day)::DATE first_day
            FROM {pairs} p JOIN numeric_owners a ON a.node_id=p.src JOIN numeric_owners b ON b.node_id=p.dst
            WHERE a.owner_code>=0 AND b.owner_code>=0 GROUP BY a.owner_code,b.owner_code\'\'\','entity_first','first_day')
        counts=np.zeros((n,6),np.int64);zsum=np.zeros(n);tails=np.zeros(n,np.int64)
        active_days=np.zeros(n,np.int16);currency=np.zeros(n,np.uint64)
        owner_n=np.zeros(ne,np.int64);owner_out=np.zeros(ne,np.int64);owner_in=np.zeros(ne,np.int64)
        day=date.fromisoformat(cfg.start);end=date.fromisoformat(cfg.end);total=0
        target=DATA_ROOT/'features/nodes';target.mkdir(parents=True,exist_ok=True)
        while day<end:
            ds=str(day);r=day_values(raw,'day',ds,['eid','src','dst','paid_z','pc','rc']);s=r['src'];d=r['dst'];total+=len(s)
            counts[:,0]+=np.bincount(s,minlength=n);counts[:,1]+=np.bincount(d,minlength=n)
            counts[:,4]+=np.bincount(s[s==d],minlength=n)
            zsum+=np.bincount(s,weights=r['paid_z'],minlength=n);tails+=np.bincount(s[np.abs(r['paid_z'])>=3],minlength=n)
            np.bitwise_or.at(currency,s,np.left_shift(np.uint64(1),r['pc'].astype(np.uint64)))
            np.bitwise_or.at(currency,d,np.left_shift(np.uint64(1),r['rc'].astype(np.uint64)))
            active=np.union1d(s,d);new=active[active_days[active]==0];own=owner[new];own=own[own>=0]
            owner_n+=np.bincount(own,minlength=ne);active_days[active]+=1
            pair=day_values(first,'first_day',ds,['src','dst'])
            counts[:,2]+=np.bincount(pair['src'],minlength=n);counts[:,3]+=np.bincount(pair['dst'],minlength=n)
            rec=day_values(reciprocal,'first_day',ds,['node','n']);counts[rec['node'],5]+=rec['n']
            ep=day_values(entity,'first_day',ds,['source_owner','target_owner'])
            owner_out+=np.bincount(ep['source_owner'],minlength=ne);owner_in+=np.bincount(ep['target_owner'],minlength=ne)
            x=np.zeros((n,19),np.float32);x[:,:6]=np.log1p(counts)
            x[:,6]=np.divide(zsum,counts[:,0],out=np.zeros(n),where=counts[:,0]>0)
            x[:,7]=np.divide(tails,counts[:,0],out=np.zeros(n),where=counts[:,0]>0)
            x[:,8]=np.log1p(np.bitwise_count(currency).astype(float));x[:,9]=np.log1p(active_days.astype(float))
            for j in range(6):x[:,10+j]=types==j
            valid=owner>=0
            for j,values in [(16,owner_n),(17,owner_out),(18,owner_in)]:x[valid,j]=np.log1p(values[owner[valid]])
            x[active_days==0]=0
            assert np.isfinite(x).all()
            np.save(target/(ds+'.npy'),x)
            if ds=='2022-09-25':
                degrees=counts[:,:2][:,::-1];occupied=degrees.sum(axis=1)>0
                if not occupied.any():raise ValueError('No training observations for degree calibration')
                delta=np.log1p(degrees[occupied]).mean(axis=0)
                if np.any(delta<=0):raise ValueError('Training graph requires incoming and outgoing observations')
                write_json(DATA_ROOT/'calibration.json',dict(delta=delta.tolist(),calibration_date=ds,
                    active_accounts=int(occupied.sum()),original_transaction_count=int(counts[:,0].sum()),fit_scope='training graph only'))
            print('daily_nodes',ds,'rows',len(s),flush=True)
            del r,pair,rec,ep,x;gc.collect();day+=timedelta(days=1)
        assert total==meta['rows']
'''


In [ ]:
RUN_INGEST = False
if RUN_INGEST:
    command("ingest")


In [ ]:
RUN_EDGE_FEATURES = False
if RUN_EDGE_FEATURES:
    command("features")


In [ ]:
RUN_DAILY_NODE_FEATURES = False
if RUN_DAILY_NODE_FEATURES:
    command("nodes")


In [ ]:
for stage in ("01_ingest", "02_edges", "03_daily_nodes"):
    marker = WORK_DIR / "data/_build" / f"{stage}.json"
    print(stage, "완료" if marker.exists() else "미완료")
metadata = WORK_DIR / "data/raw_metadata.json"
if metadata.exists():
    value = json.loads(metadata.read_text())
    print("관측 거래:", f"{value['rows']:,}", "계좌:", f"{value['nodes']:,}", "양성:", f"{value['positive']:,}")
